# 🎓 College Buddy - Google Colab Setup 🚀

This notebook runs the College Buddy Chatbot Backend on Google's T4 GPU.

### Features:
- **Instant Responses**: Uses T4 GPU for <2s inference.
- **Model**: Uses `llama3.2:3b` for better reasoning.
- **Public URL**: Exposes the backend via Ngrok so you can connect your frontend.

### ⚠️ IMPORTANT: Runtime Type
Make sure you are using a GPU Runtime:
1. Click **Runtime** > **Change runtime type**
2. Select **T4 GPU**
3. Click **Save**

In [ ]:
# Verify GPU is available
!nvidia-smi

## 1. Install Dependencies
Installing Ollama, LangChain, ChromaDB, and other required libraries.

In [ ]:
# Install Ollama (Linux version for Colab)
!curl -fsSL https://ollama.com/install.sh | sh

# Fix for ChromaDB in Colab (needs newer SQLite)
!pip install pysqlite3-binary
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# Install Python Dependencies
!pip install fastapi uvicorn pyngrok nest_asyncio sentence-transformers chromadb langchain langchain-community langchain-ollama pandas openpyxl

## 2. Start Ollama and Pull Model
Starting the Ollama server in the background and pulling `llama3.2:3b`.

In [ ]:
import subprocess
import time
import os

# Check if Ollama is installed
if not os.path.exists('/usr/local/bin/ollama') and not os.path.exists('/usr/bin/ollama'):
    print("❌ Ollama executable not found! Did you run the 'Install Dependencies' cell above?")
else:
    print("✅ Ollama found, starting server...")
    
    # Start Ollama Serve in background
    # Redirect output to /dev/null to keep cell clean
    process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)  # Wait for it to start

    print("⏳ Pulling llama3.2:3b model (this may take a few minutes)...")
    !ollama pull llama3.2:3b
    print("✅ Model pulled successfully!")

## 3. Upload Codebase
Please upload your local files to the Colab environment:
1. Click the **Files** folder icon on the left sidebar.
2. Drag and drop your **`app`** folder and **`backend.py`** file.

*(Note: Ensure `app/` is uploaded as a folder. If you zip it, unzip it below)*

In [ ]:
# If you uploaded a zip file (e.g., code.zip), uncomment to unzip:
# !unzip code.zip

print("Checking file structure...")
!ls -R | grep ":$" | head -n 5

# Verify critical files exist
import os
if os.path.exists('backend.py') and os.path.exists('app'):
    print("✅ Codebase looks good!")
else:
    print("❌ WARNING: 'backend.py' or 'app/' folder missing. Please upload them.")

## 4. Start Backend Server with Public URL
This will give you a public URL (e.g., `https://xyz.ngrok-free.app`) to use in your Frontend.

In [ ]:
from pyngrok import ngrok
import uvicorn
import nest_asyncio
import threading

# Authenticate Ngrok (You need a free token from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = "38AFZ9ehRUeryGuOWMeyCuQ4dvm_3yEgMYUjpWJsY4hTChMVf"  # <--- UPDATE THIS IF NEEDED
ngrok.set_auth_token(NGROK_TOKEN)

# Kill existing tunnels
ngrok.kill()

# Start Tunnel
try:
    public_url = ngrok.connect(8000).public_url
    print(f"🚀 Public URL: {public_url}")
    print(f"📋 Copy this URL to your frontend's API_BASE_URL")
except Exception as e:
    print(f"❌ Ngrok Error: {e}")

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

# Run Server
try:
    from backend import app
    # Run on 0.0.0.0 to bind to all interfaces
    uvicorn.run(app, host="0.0.0.0", port=8000)
except ImportError:
    print("❌ ERROR: 'backend.py' not found! Please upload your code manually.")
except Exception as e:
    print(f"❌ Server Error: {e}")